# Model Router Toolkit — Quickstart

**Two models are better than one.** Defaulting to a single frontier model for all queries overpays for routine tasks and settles for suboptimal results on specialized ones. The counter-intuitive truth: combining models with intelligent routing surpasses the accuracy of any single model while dramatically lowering cost.

The Model Router uses ML-driven confidence prediction to select the right model for every query in under 10ms. 63% of queries can be handled by efficient models without accuracy loss — frontier models kick in only for the queries that need them. The result: **50%+ cost savings at frontier-level quality**, with a pipeline that improves as it sees more data.

This notebook demonstrates routing end-to-end — no servers, no deployment, no heavy dependencies. Just an API key and a pre-trained routing model.

---
## How It Works

The routing pipeline makes an intelligent decision in four stages — all before any LLM is called:

```
  ┌──────────┐     ┌──────────┐     ┌───────────┐     ┌──────────┐
  │  Embed   │ ──▶ │ Cluster  │ ──▶ │  Estimate │ ──▶ │  Route   │
  │          │     │          │     │           │     │          │
  │ Convert  │     │ Find the │     │ Confidence│     │ Select   │
  │ query to │     │ query's  │     │ prediction│     │ optimal  │
  │ a vector │     │ topic    │     │ per model │     │ model    │
  │          │     │ region   │     │           │     │          │
  │ (~50ms)  │     │ (<1ms)   │     │  (<1ms)   │     │  (<1ms)  │
  └──────────┘     └──────────┘     └───────────┘     └──────────┘
```

**Embed:** Convert the query into a semantic vector via an embedding API.

**Cluster:** Assign the query to one of 100 topic regions learned from evaluation data.

**Estimate:** Look up calibrated P(correct) for each model on this topic — confidence prediction tells us which models will get this right.

**Route:** Pick the most cost-efficient model whose confidence is within tolerance of the best. Routine queries go to fast, cheap models. Hard queries escalate to frontier.

### The Model Pool

Different models excel at different tasks. The router combines them so you get the best of each:

| Model | Strength | Cost per 1k Queries |
|-------|----------|--------------------:|
| Nemotron 3 Nano | Fast, direct answers | $0.04 |
| Nemotron 3 Nano Think | Chain-of-thought reasoning | $0.23 |
| GPT-OSS 20B | High-quality reasoning | $0.39 |

The bundled router covers these 3 models. The [advanced path](../README.md) adds Nemotron 3 Super, GPT-OSS 120B, Qwen 3.5 122B, GPT-5.2, and Claude Opus 4.6 — the more models in the pool, the better accuracy *and* cost.

---
## Setup

Install dependencies (skip if already installed) and set your API key.

In [1]:
%pip install -q requests numpy scikit-learn


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: /opt/homebrew/opt/python@3.11/bin/python3.11 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os

NVIDIA_API_KEY = "REDACTED_API_KEY"
if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = input("Enter your NVIDIA API key (get one at https://build.nvidia.com/): ")
    os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY

print(f"API key set ({len(NVIDIA_API_KEY)} chars).")

API key set (70 chars).


---
## Load the Router

This loads the pre-trained KMeans router from a pickle file. No GPU, no torch — just sklearn.

In [3]:
import pickle
import numpy as np
from pathlib import Path

PKL_PATH = Path("../checkpoints/kmeans_c100_db.pkl")
with open(PKL_PATH, "rb") as f:
    db = pickle.load(f)

kmeans_model = db["kmeans_model"]
platt_models = db["platt_models"]
cluster_acc = db["cluster_acc"]
models = db["models"]
split_models = db["split_models"]
model_to_split = dict(zip(models, split_models))
n_clusters = db["n_clusters"]

print(f"Loaded: {n_clusters} clusters, {len(models)} models")
print(f"Models: {models}")

Loaded: 100 clusters, 3 models
Models: ['gptoss-high', 'nem-think', 'nem-nothink']


---
## Define Model Costs and API Mappings

In [4]:
NVIDIA_BASE_URL = "https://integrate.api.nvidia.com/v1"
EMBED_MODEL = "nvidia/llama-nemotron-embed-1b-v2"

MODEL_CONFIG = {
    "nem-nothink": {
        "display": "Nemotron 3 Nano",
        "api_model": "nvidia/nemotron-3-nano-30b-a3b",
        "cost_per_1k": 0.04,
        "system_prompt": "Answer directly and concisely.",
    },
    "nem-think": {
        "display": "Nemotron 3 Nano Think",
        "api_model": "nvidia/nemotron-3-nano-30b-a3b",
        "cost_per_1k": 0.23,
        "system_prompt": "Think step-by-step before answering.",
        "extra_body": {"thinking": {"type": "enabled", "budget_tokens": 4096}},
    },
    "gptoss-high": {
        "display": "GPT-OSS 20B",
        "api_model": "openai/gpt-oss-20b",
        "cost_per_1k": 0.39,
        "system_prompt": "You are a helpful assistant. Think carefully before answering.",
    },
}

MODELS_BY_COST = sorted(MODEL_CONFIG.keys(), key=lambda m: MODEL_CONFIG[m]["cost_per_1k"])

---
## Embedding and Routing Functions

In [5]:
import requests
import time

def embed(text):
    """Embed text via build.nvidia.com API."""
    resp = requests.post(
        f"{NVIDIA_BASE_URL}/embeddings",
        headers={"Authorization": f"Bearer {NVIDIA_API_KEY}", "Content-Type": "application/json"},
        json={"model": EMBED_MODEL, "input": text, "input_type": "query", "encoding_format": "float"},
        timeout=30,
    )
    resp.raise_for_status()
    return np.array(resp.json()["data"][0]["embedding"], dtype=np.float32)


def route(embedding, tolerance=0.20):
    """Route an embedding to the most cost-efficient model above threshold."""
    cluster = int(kmeans_model.predict(embedding.reshape(1, -1))[0])

    probs = {}
    for model in models:
        split_key = model_to_split.get(model, model)
        raw_acc = cluster_acc.get(cluster, {}).get(split_key, 0.5)
        if model in platt_models:
            probs[model] = float(platt_models[model].predict_proba([[raw_acc]])[0, 1])
        else:
            probs[model] = raw_acc

    p_max = max(probs.values())
    threshold = p_max - tolerance

    selected = MODELS_BY_COST[-1]
    for m in MODELS_BY_COST:
        if probs.get(m, 0) >= threshold:
            selected = m
            break

    return {
        "selected_model": selected,
        "probs": probs,
        "cluster": cluster,
        "p_max": p_max,
        "cost_per_query": MODEL_CONFIG[selected]["cost_per_1k"] / 1000,
    }

---
## Try It: Route Two Questions

Let's send an easy question and a hard question through the router.

In [6]:
def route_and_display(question, tolerance=0.20):
    t0 = time.time()
    emb = embed(question)
    embed_ms = (time.time() - t0) * 1000

    t0 = time.time()
    result = route(emb, tolerance=tolerance)
    route_ms = (time.time() - t0) * 1000

    selected = result["selected_model"]
    cost_1k = MODEL_CONFIG[selected]["cost_per_1k"]

    print(f'Question: "{question}"')
    print(f"Cluster:  {result['cluster']}")
    print(f"Latency:  {embed_ms:.0f}ms embed + {route_ms:.1f}ms route\n")

    print("Model scores (calibrated probability of correct answer):")
    for m in ["gptoss-high", "nem-think", "nem-nothink"]:
        if m not in result["probs"]:
            continue
        p = result["probs"][m]
        bar = "\u2588" * int(p * 30) + "\u2591" * (30 - int(p * 30))
        c = MODEL_CONFIG[m]["cost_per_1k"]
        marker = " << SELECTED" if m == selected else ""
        print(f"  {MODEL_CONFIG[m]['display']:<22} {p:.3f} {bar} ${c:.2f}/1k{marker}")

    most_expensive = max(MODEL_CONFIG.values(), key=lambda c: c["cost_per_1k"])
    savings = (1 - cost_1k / most_expensive["cost_per_1k"]) * 100 if most_expensive["cost_per_1k"] > 0 else 0
    print(f"\nRouted to {MODEL_CONFIG[selected]['display']} at ${cost_1k:.2f}/1k queries (saving {savings:.0f}% vs {most_expensive['display']})")
    return result

In [7]:
print("=" * 70)
print("EXAMPLE 1: Simple factual question")
print("=" * 70)
result_easy = route_and_display("What is the capital of France?")

EXAMPLE 1: Simple factual question
Question: "What is the capital of France?"
Cluster:  49
Latency:  647ms embed + 30.0ms route

Model scores (calibrated probability of correct answer):
  GPT-OSS 20B            0.641 ███████████████████░░░░░░░░░░░ $0.39/1k
  Nemotron 3 Nano Think  0.764 ██████████████████████░░░░░░░░ $0.23/1k << SELECTED
  Nemotron 3 Nano        0.651 ███████████████████░░░░░░░░░░░ $0.04/1k

Routed to Nemotron 3 Nano Think at $0.23/1k queries (saving 41% vs GPT-OSS 20B)


In [8]:
print("=" * 70)
print("EXAMPLE 2: Complex reasoning question")
print("=" * 70)
result_hard = route_and_display("Prove that the square root of 2 is irrational")

EXAMPLE 2: Complex reasoning question
Question: "Prove that the square root of 2 is irrational"
Cluster:  84
Latency:  628ms embed + 1.1ms route

Model scores (calibrated probability of correct answer):
  GPT-OSS 20B            0.556 ████████████████░░░░░░░░░░░░░░ $0.39/1k
  Nemotron 3 Nano Think  0.722 █████████████████████░░░░░░░░░ $0.23/1k << SELECTED
  Nemotron 3 Nano        0.604 ██████████████████░░░░░░░░░░░░ $0.04/1k

Routed to Nemotron 3 Nano Think at $0.23/1k queries (saving 41% vs GPT-OSS 20B)


---
## Call the Selected Model

Now let's actually call the routed models and see their responses.

In [9]:
def call_model(question, route_result):
    """Call the selected model via build.nvidia.com chat completions API."""
    selected = route_result["selected_model"]
    cfg = MODEL_CONFIG[selected]

    print(f"Calling {cfg['display']}...\n")

    body = {
        "model": cfg["api_model"],
        "messages": [
            {"role": "system", "content": cfg["system_prompt"]},
            {"role": "user", "content": question},
        ],
        "temperature": 0.7,
        "max_tokens": 1024,
        "stream": False,
    }
    if "extra_body" in cfg:
        body.update(cfg["extra_body"])

    t0 = time.time()
    resp = requests.post(
        f"{NVIDIA_BASE_URL}/chat/completions",
        headers={"Authorization": f"Bearer {NVIDIA_API_KEY}", "Content-Type": "application/json"},
        json=body,
        timeout=60,
    )
    latency = time.time() - t0
    resp.raise_for_status()
    data = resp.json()

    answer = data["choices"][0]["message"]["content"]
    usage = data.get("usage", {})

    print(f"Answer:\n{answer[:1000]}")
    print(f"\nTokens: {usage.get('prompt_tokens', '?')} prompt + {usage.get('completion_tokens', '?')} completion")
    print(f"Latency: {latency:.1f}s")
    return answer

In [10]:
print("=" * 70)
print("EXAMPLE 1 RESPONSE")
print("=" * 70)
answer_easy = call_model("What is the capital of France?", result_easy)

EXAMPLE 1 RESPONSE
Calling Nemotron 3 Nano Think...

Answer:

The capital of France is **Paris**.

Tokens: 30 prompt + 35 completion
Latency: 0.5s


In [11]:
print("=" * 70)
print("EXAMPLE 2 RESPONSE")
print("=" * 70)
answer_hard = call_model("Prove that the square root of 2 is irrational", result_hard)

EXAMPLE 2 RESPONSE
Calling Nemotron 3 Nano Think...

Answer:

**Theorem.**  \(\displaystyle \sqrt{2}\) is not a rational number; i.e. there are no integers \(p,q\) with \(q\neq 0\) such that \(\sqrt{2}=p/q\).

---

### Proof (by contradiction)

1. **Assume the opposite.**  
   Suppose \(\sqrt{2}\) *is* rational. Then we can write it as a fraction in lowest terms, i.e.  

   \[
   \sqrt{2}= \frac{p}{q},
   \qquad p,q\in\mathbb Z,\; q\neq 0,
   \]

   where \(p\) and \(q\) have no common divisor greater than 1 (they are *coprime*).

2. **Square both sides.**  
   \[
   2 = \left(\frac{p}{q}\right)^{2}
   \;\Longrightarrow\;
   2q^{2}=p^{2}.
   \tag{1}
   \]

3. **Deduce that \(p\) must be even.**  
   The left‑hand side of (1) is an even integer (it is \(2\) times the integer \(q^{2}\)).  
   Hence \(p^{2}\) is even.  
   If a square of an integer is even, the integer itself must be even (because the parity of a product is determined by each factor).  
   Therefore there exists an intege

---
## Results

The router automatically matched each query to the right model — cheap models for routine tasks, stronger models only when needed.

In [12]:
sel_easy = result_easy["selected_model"]
sel_hard = result_hard["selected_model"]
most_expensive = max(MODEL_CONFIG.keys(), key=lambda m: MODEL_CONFIG[m]["cost_per_1k"])
exp_name = MODEL_CONFIG[most_expensive]["display"]
exp_cost = MODEL_CONFIG[most_expensive]["cost_per_1k"]

rows = [
    ("Cluster", str(result_easy["cluster"]), str(result_hard["cluster"])),
    ("Selected Model", MODEL_CONFIG[sel_easy]["display"], MODEL_CONFIG[sel_hard]["display"]),
    ("p(correct)", f"{result_easy['probs'][sel_easy]:.3f}", f"{result_hard['probs'][sel_hard]:.3f}"),
    ("Best model p(correct)", f"{result_easy['p_max']:.3f}", f"{result_hard['p_max']:.3f}"),
    ("Cost per 1k queries", f"${MODEL_CONFIG[sel_easy]['cost_per_1k']:.2f}", f"${MODEL_CONFIG[sel_hard]['cost_per_1k']:.2f}"),
    ("Most expensive option", f"${exp_cost:.2f}", f"${exp_cost:.2f}"),
]

print(f"{'':30} {'Easy Question':>20} {'Hard Question':>20}")
print("-" * 72)
for label, easy, hard in rows:
    print(f"{label:<30} {easy:>20} {hard:>20}")

total_routed = result_easy["cost_per_query"] + result_hard["cost_per_query"]
total_best = 2 * exp_cost / 1000
savings = (1 - total_routed / total_best) * 100
print(f"\nTotal cost (2 queries): ${total_routed * 1000:.2f}/1k vs ${total_best * 1000:.2f}/1k always using {exp_name}")
print(f"Savings: {savings:.0f}%")

                                      Easy Question        Hard Question
------------------------------------------------------------------------
Cluster                                          49                   84
Selected Model                 Nemotron 3 Nano Think Nemotron 3 Nano Think
p(correct)                                    0.764                0.722
Best model p(correct)                         0.764                0.722
Cost per 1k queries                           $0.23                $0.23
Most expensive option                         $0.39                $0.39

Total cost (2 queries): $0.46/1k vs $0.78/1k always using GPT-OSS 20B
Savings: 41%


---
## What's Next

This demo used a lightweight KMeans router. The toolkit supports two paths for going further:

### Deploy as a service (30 min)

Stand up an OpenAI-compatible endpoint that any application can point to:

```bash
pip install -e .
model-router setup        # detect environment, configure model pool
model-router serve         # start server at localhost:8000/v1/chat/completions
```

### Plug into an existing LiteLLM app (3 lines)

```python
from litellm import Router
from model_router_toolkit import ModelRoutingStrategy

router = Router(model_list=my_models)
strategy = ModelRoutingStrategy.from_config("pool_config.yaml")
router.set_custom_routing_strategy(strategy)
```

### Train on your own data (continuously improve)

The data flywheel: collect production data, retrain the router, get better routing, collect more data.

```bash
model-router collect --config pool.yaml --questions questions.txt --output data/train.csv
model-router train --config pool.yaml --data data/train.csv --output-dir checkpoints/custom/
model-router evaluate --checkpoint checkpoints/custom/router.pkl --data data/test.csv
```

See the [README](../README.md) for the full journey.